# GAN — Renkli Giysi Tasarımı Üretimi

## Sıfırdan, Adım Adım Öğreniyoruz

---

## GAN Nedir? Neden Bu Kadar Özel?

Şimdiye kadar öğrendiğin tüm modeller **sınıflandırıcıydı**:

- FNN → "Bu müşteri ayrılacak mı?"
- CNN → "Bu resim ayakkabı mı, çanta mı?"
- RNN → "Yarın hisse fiyatı ne olur?"

Bunlar hepsi mevcut bir şeyi **analiz ediyor**.

**GAN ise tamamen farklı bir şey yapıyor: yoktan bir şey yaratıyor.**

GAN'a şunu sorabilirsin:

- "Bana hiç var olmamış bir insan yüzü çiz"
- "Hiç kimsenin görmediği bir giysi tasarla"
- "Sahte ama gerçekçi bir ev fotoğrafı üret"

Ve GAN bunu yapabiliyor. Peki nasıl?

---

## Temel Fikir: İki Model, Bir Yarışma

GAN'ın içinde **birbirine karşı yarışan** iki model var:

### 🎨 Generator (Üretici) — Sahtekâr

Hiçbir şeyden (saf rastgele gürültüden) gerçekçi resimler üretmeye çalışır.  
Amacı: Discriminator'ı kandırmak.

### 🔍 Discriminator (Ayırt Edici) — Dedektif

Gelen her resmin gerçek mi yoksa Generator'ın ürettiği sahte mi olduğunu anlamaya çalışır.  
Amacı: Generator'ı yakalamak.

```
┌──────────────────────────────────────────────────────────────┐
│                                                              │
│   [Rastgele Gürültü] ──→ GENERATOR ──→ [Sahte Resim]        │
│                                               │              │
│   [Gerçek Resim] ─────────────────────────→  ↓              │
│                                          DISCRIMINATOR       │
│                                               │              │
│                                     "Gerçek mi? Sahte mi?"   │
└──────────────────────────────────────────────────────────────┘
```

### Gerçek Hayat Analojisi 💡

**Sahte para basan biri (Generator)** ve **para sahteciliği dedektifi (Discriminator)** arasındaki yarışı düşün:

1. Sahtekâr kötü sahte para basar → Dedektif hemen yakalar
2. Sahtekâr yakalanınca daha iyi sahte para basmayı öğrenir
3. Dedektif daha iyi sahteyi görünce daha dikkatli olmayı öğrenir
4. Bu döngü devam eder...
5. Sonunda sahtekâr o kadar iyi olur ki sahteyi gerçekten ayırt etmek imkânsızlaşır

İşte GAN tam olarak bu!

---

## Bu Notebook'ta Ne Yapacağız?

Fashion MNIST veri setindeki giysi resimlerini öğrenerek **tamamen yeni, renkli giysi tasarımları** üreteceğiz.

Adımlar:

1. Veriyi yükle ve renklendir
2. Generator modelini oluştur
3. Discriminator modelini oluştur
4. İkisini birlikte eğit
5. Yeni giysiler üret!


In [ ]:
# Gerekli paketleri kur
!pip install tensorflow numpy matplotlib -q
print("Kurulum tamamlandı.")

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt
import time

# random_seed: her çalıştırmada aynı sonuçları almak için
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow sürümü: {tf.__version__}")
print(f"GPU var mı: {len(tf.config.list_physical_devices('GPU')) > 0}")
print("\nNot: GPU varsa ~15 dk, CPU'da ~45-60 dk sürebilir.")

# --- nn3d: agi tarayicida canli 3D izlemek icin ---------------------------
import sys, pathlib
if not any(pathlib.Path(p, "nn3d").is_dir() for p in sys.path):
    sys.path.insert(0, str(pathlib.Path.cwd().parents[1] / "src"))
import nn3d


---

## Adım 1: Veriyi Yükle ve Renklendir

### Fashion MNIST Nedir?

Zalando şirketinin hazırladığı 70.000 giysi fotoğrafından oluşan bir veri seti.  
Her resim **28×28 piksel** ve **gri tonlamalı** (tek kanal, renkli değil).

10 kategori var: T-Shirt, Pantolon, Kazak, Elbise, Mont, Sandalet, Gömlek, Spor Ayakkabı, Çanta, Bot

### Sorun: Gri Tonlamalı

GAN'ımızın renkli çıktı vermesini istiyoruz. Ama veri gri.  
Çözüm: Her kategoriye bir renk atayıp 3 kanallı (RGB) veriye çeviriyoruz.

```
Gri piksel değeri: 0.8
         ×
T-Shirt rengi: [0.95, 0.20, 0.20]   (Kırmızı)
         =
RGB piksel:    [0.76, 0.16, 0.16]   → Koyu kırmızı piksel
```

### Normalizasyon: [-1, +1]

Diğer modellerde veriyi [0, 1] aralığına çektik.  
GAN'da ise **[-1, +1]** kullanıyoruz.

Neden? Generator'ın son katmanında `tanh` aktivasyonu var.  
Tanh'ın çıkışı zaten [-1, +1] arasındadır.  
Veri ile Generator çıkışı aynı aralıkta olursa karşılaştırma daha doğru olur.

```
Dönüşüm formülü: yeni_değer = (eski_değer × 2) - 1

0.0  →  -1.0  (siyah piksel)
0.5  →   0.0  (gri piksel)
1.0  →  +1.0  (beyaz piksel)
```


In [ ]:
CLASS_NAMES = ['T-Shirt', 'Pantolon', 'Kazak', 'Elbise', 'Mont',
               'Sandalet', 'Gömlek', 'Spor Ayakkabı', 'Çanta', 'Bot']

# Her kategori için RGB renk [Kırmızı, Yeşil, Mavi] — 0.0 ile 1.0 arası
CLASS_COLORS = np.array([
    [0.95, 0.20, 0.20],  # T-Shirt        → Kırmızı
    [0.15, 0.25, 0.80],  # Pantolon       → Koyu Mavi
    [0.95, 0.55, 0.10],  # Kazak          → Turuncu
    [0.80, 0.20, 0.75],  # Elbise         → Mor
    [0.15, 0.65, 0.25],  # Mont           → Yeşil
    [0.92, 0.78, 0.10],  # Sandalet       → Altın Sarısı
    [0.25, 0.70, 0.95],  # Gömlek         → Açık Mavi
    [0.95, 0.40, 0.10],  # Spor Ayakkabı  → Turuncu-Kırmızı
    [0.65, 0.25, 0.88],  # Çanta          → Violet
    [0.50, 0.28, 0.12],  # Bot            → Kahverengi
], dtype=np.float32)


def colorize(images_gray, labels):
    """
    Gri resimleri kategoriye göre renklendirir.
    
    Girdi:  images_gray (N, 28, 28) — gri piksel değerleri
            labels      (N,)        — her resmin kategorisi (0-9)
    Çıktı:  colored     (N, 28, 28, 3) — renkli RGB resimler
    """
    colored = np.zeros((len(images_gray), 28, 28, 3), dtype=np.float32)
    for i in range(len(images_gray)):
        renk = CLASS_COLORS[labels[i]]                    # Bu kategorinin rengi (3,)
        colored[i] = images_gray[i, :, :, None] * renk   # (28,28,1) × (3,) = (28,28,3)
    return colored


# Veriyi indir (ilk seferde internet gerekir)
print("Fashion MNIST yükleniyor...")
(X_raw, y_train), _ = tf.keras.datasets.fashion_mnist.load_data()

# 1. Adım: [0, 255] → [0.0, 1.0]
X_gray = X_raw.astype(np.float32) / 255.0

# 2. Adım: Renklendirme
print("Renkli veriye dönüştürülüyor...")
X_color = colorize(X_gray, y_train)   # (60000, 28, 28, 3)

# 3. Adım: [0.0, 1.0] → [-1.0, +1.0]  (GAN normalizasyonu)
X_color = (X_color * 2.0) - 1.0

print(f"\nVeri boyutu: {X_color.shape}")
print(f"  → 60.000 resim, 28×28 piksel, 3 renk kanalı (RGB)")
print(f"Değer aralığı: [{X_color.min():.1f}, {X_color.max():.1f}]")

In [ ]:
# Her kategoriden bir örnek göster
fig, axes = plt.subplots(2, 5, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    idx = np.where(y_train == i)[0][5]
    img = (X_color[idx] + 1) / 2        # Görselleştirme için [-1,+1] → [0,1]
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[i], fontsize=10, fontweight='bold')
    ax.axis('off')

plt.suptitle('Renklendirilmiş Eğitim Verisi — GAN Bunları Taklit Etmeyi Öğrenecek',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Eğitim için veri akışı oluştur
BATCH_SIZE = 128
# Batch Size: Her eğitim adımında kaç resim işleneceği
# 128 resim → ağırlıkları güncelle → 128 resim daha → güncelle → ...

dataset = (
    tf.data.Dataset.from_tensor_slices(X_color)
    .shuffle(60000)               # Her epoch'ta veriyi karıştır
    .batch(BATCH_SIZE, drop_remainder=True)  # 128'lik batch'lere böl
    .prefetch(tf.data.AUTOTUNE)   # CPU veriyi hazırlarken GPU eğitsin
)

print(f"Toplam batch sayısı: {len(dataset)}")
print(f"Yani her epoch'ta {len(dataset)} × {BATCH_SIZE} = {len(dataset) * BATCH_SIZE} resim işlenecek")

---

## Adım 2: Generator — Sahtekâr Model

### Generator Ne Yapar?

Generator rastgele bir sayı dizisi alır (buna **latent vector** veya **gürültü** diyoruz) ve bundan gerçek gibi görünen bir resim üretir.

```
[0.32, -1.45, 0.78, ..., -0.21]   ← 256 rastgele sayı (latent vector)
              ↓
         GENERATOR
              ↓
    [28×28 piksel, 3 kanal]        ← Renkli giysi resmi
```

### Latent Vector (Gizli Vektör) Nedir?

256 boyutlu bir vektör. Her farklı vektör → farklı bir resim.

```
Vektör A: [0.32, -1.45, ...]  →  Kırmızı t-shirt
Vektör B: [-0.91, 0.67, ...]  →  Mavi pantolon
Vektör C: [1.23, -0.34, ...]  →  Mor elbise
```

GAN eğitim boyunca "hangi vektör → hangi giysi" ilişkisini öğrenir.

### Generator Mimarisi: Küçükten Büyüğe

Generator küçük bir tensörden başlayarak giderek büyütür (upsampling):

```
Gürültü: (256,)                  ← 256 sayı
    ↓  Dense katman
(7 × 7 × 512) = 25.088 sayı      ← Küçük ama derin özellik haritası
    ↓  Conv2DTranspose (stride=2)
(14 × 14 × 256)                  ← 2× büyüdü
    ↓  Conv2DTranspose (stride=2)
(28 × 28 × 128)                  ← Tekrar 2× büyüdü
    ↓  Conv2DTranspose (stride=1)
(28 × 28 × 3)                    ← Final: 28×28 piksel, RGB!
```

### Conv2DTranspose Nedir? (Ters Konvolüsyon)

CNN'de öğrendiğin `Conv2D` görüntüyü **küçülten** bir işlemdi.  
`Conv2DTranspose` ise tam tersi — görüntüyü **büyüten** işlem.

```
Conv2D          (stride=2): 28×28  →  14×14  (küçültür)
Conv2DTranspose (stride=2): 14×14  →  28×28  (büyütür)
```

Bu yüzden Generator'da Conv2DTranspose kullanıyoruz — küçük özellik haritasından büyük resim üretmek için.

### LeakyReLU Nedir?

Normal ReLU'yu hatırlıyor musun? `max(0, x)` — negatif değerleri tamamen sıfırlar.  
Bu GAN'larda sorun çıkarır: negatif bölgelerdeki nöronlar tamamen ölür ve öğrenmeyi durdurur.

```
ReLU:      x = -5  →  çıkış: 0      (tamamen sıfır, gradyan yok!)
LeakyReLU: x = -5  →  çıkış: -1    (0.2 × -5, küçük ama var)
```

LeakyReLU negatif değerlere 0.2 eğimi verir → nöronlar canlı kalır → daha iyi öğrenir.

### BatchNormalization Nedir?

Her katmandan çıkan değerleri normalize eder (ortalama ≈ 0, standart sapma ≈ 1).  
Bunu yapmak eğitimi çok daha hızlı ve kararlı hale getirir.  
GAN'larda BatchNormalization olmadan eğitim genellikle çöküyor.

### Neden Çıkışta `tanh`?

Tanh aktivasyonu çıkışı **[-1, +1]** arasına sıkıştırır.  
Verilerimizi de [-1, +1] aralığına normalize etmiştik.  
Generator'ın çıkışı ile gerçek veri aynı aralıkta olduğunda karşılaştırma adil olur.


In [ ]:
LATENT_DIM = 256   # Gürültü vektörünün boyutu — büyük = daha çeşitli tasarımlar

def build_generator():
    return models.Sequential([

        # === 1. AŞAMA: Gürültüyü özellik haritasına dönüştür ===
        # 256 sayıyı → 7×7×512 = 25.088 sayıya genişlet
        layers.Dense(7 * 7 * 512, use_bias=False, input_shape=(LATENT_DIM,)),
        layers.BatchNormalization(),   # Değerleri normalize et → kararlı eğitim
        layers.LeakyReLU(0.2),         # Negatif değerlere 0.2 eğimi ver
        layers.Reshape((7, 7, 512)),   # Düz vektörü 3D özellik haritasına dönüştür
        # Şu an: (7, 7, 512)

        # === 2. AŞAMA: 7×7 → 14×14 (2x büyütme) ===
        # stride=2 → her adımda 2 piksel atlayarak büyütür
        layers.Conv2DTranspose(256, (4, 4), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        # Şu an: (14, 14, 256)

        # === 3. AŞAMA: 14×14 → 28×28 (2x büyütme) ===
        layers.Conv2DTranspose(128, (4, 4), strides=(2, 2), padding='same', use_bias=False),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        # Şu an: (28, 28, 128)

        # === 4. AŞAMA: Son katman — RGB resim çıkışı ===
        # stride=1 → boyut değişmez, sadece kanal sayısı 128 → 3 olur (RGB)
        # activation='tanh' → çıkışı [-1, +1] aralığına sıkıştırır
        layers.Conv2DTranspose(3, (4, 4), strides=(1, 1), padding='same',
                               use_bias=False, activation='tanh'),
        # Final çıkış: (28, 28, 3) → Renkli giysi resmi!

    ], name='Generator')


generator = build_generator()
generator.summary()

# nn3d: Generator'u canlı izle. GAN'in kendi eğitim döngüsü var
# (model.fit yok), bu yüzden Monitor yerine view.update() kullanıyoruz.
gorunum = nn3d.show(
    generator,
    sample=tf.random.normal([1, LATENT_DIM]).numpy(),
    name='GAN Generator',
)

print("\nÖnemli sütunlar:")
print("Output Shape → Her katmanın çıkış boyutu")
print("Param #      → Her katmandaki öğrenilecek parametre (ağırlık) sayısı")

In [ ]:
# Eğitilmemiş Generator ne üretiyor? (saf rastgele gürültü)
giris_gurultusu = tf.random.normal([5, LATENT_DIM])   # 5 farklı vektör
sahte_resimler  = generator(giris_gurultusu, training=False)

fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for i, ax in enumerate(axes):
    img = np.clip((sahte_resimler[i].numpy() + 1) / 2, 0, 1)  # [-1,+1] → [0,1]
    ax.imshow(img)
    ax.set_title(f'Gürültü #{i+1}', fontsize=9)
    ax.axis('off')

plt.suptitle('Eğitim Öncesi Generator Çıkışı — Tamamen Anlamsız Renkli Gürültü',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()
print("Eğitim sonunda bu karelerde giysiler göreceksin!")

---

## Adım 3: Discriminator — Dedektif Model

### Discriminator Ne Yapar?

Bir resim alır ve şunu söyler: **"Bu gerçek mi, yoksa Generator'ın ürettiği sahte mi?"**

```
Gerçek giysi fotoğrafı  →  DISCRIMINATOR  →  Yüksek sayı (gerçek)
Generator'ın sahtesi    →  DISCRIMINATOR  →  Düşük sayı (sahte)
```

### Discriminator Mimarisi: Büyükten Küçüğe

CNN'e benzer — resmi sıkıştırır, özellik çıkarır:

```
(28, 28, 3) ← Giriş: Renkli resim
    ↓  Conv2D (stride=2)
(14, 14, 64)
    ↓  Conv2D (stride=2)
(7, 7, 128)
    ↓  Conv2D (stride=2)
(4, 4, 256)
    ↓  Flatten
(4096,)
    ↓  Dense
(1,)   ← Tek sayı: büyük = gerçek, küçük = sahte
```

### Neden Discriminator'ın Çıkışında Aktivasyon Yok?

Normalde sınıflandırmada Sigmoid kullanırdık (çıkışı 0-1 yapmak için).  
Ama burada sigmoid'i kayıp fonksiyonunun içinde hesaplatıyoruz (`from_logits=True`).  
Sebebi: Sayısal olarak çok daha kararlı bir hesaplama yapılıyor.

Ham sayı (logit) → Kayıp fonksiyonu içinde sigmoid → Kayıp değeri

### Dropout Neden Var?

Discriminator Generator'dan çok daha hızlı öğrenirse bir sorun çıkar:  
Discriminator her sahtede %99.9 kesinlikle "sahte" der → Generator'a hiç bilgi kalmaz → Generator öğrenemez.

Dropout ile Discriminator'ı biraz yavaşlatıyoruz → İki model dengeli gelişiyor.


In [ ]:
def build_discriminator():
    return models.Sequential([

        # === 1. AŞAMA: 28×28 → 14×14 ===
        # Giriş: (28, 28, 3) — renkli resim
        layers.Conv2D(64, (4, 4), strides=(2, 2), padding='same', input_shape=(28, 28, 3)),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),   # %30 nöronu kapat → Discriminator'ı yavaşlat
        # Şu an: (14, 14, 64)

        # === 2. AŞAMA: 14×14 → 7×7 ===
        layers.Conv2D(128, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        layers.Dropout(0.3),
        # Şu an: (7, 7, 128)

        # === 3. AŞAMA: 7×7 → 4×4 ===
        layers.Conv2D(256, (4, 4), strides=(2, 2), padding='same'),
        layers.BatchNormalization(),
        layers.LeakyReLU(0.2),
        # Şu an: (4, 4, 256)

        # === 4. AŞAMA: Sonuç ===
        layers.Flatten(),   # (4, 4, 256) → (4096,) düzleştir
        layers.Dense(1),    # Tek sayı: büyük pozitif = gerçek, büyük negatif = sahte
        # NOT: Aktivasyon YOK — from_logits=True ile kayıp içinde sigmoid hesaplanacak

    ], name='Discriminator')


discriminator = build_discriminator()
discriminator.summary()

---
## Adım 4: Kayıp Fonksiyonları — Modeller Nasıl "Yanılıyor"?

### Genel Kavram: Kayıp (Loss) Nedir?

Kayıp fonksiyonu modelin ne kadar yanlış yaptığını ölçer.
Eğitim bu değeri minimize etmeye (küçültmeye) çalışır.

GAN'da **iki ayrı kayıp** var — bir tane Generator için, bir tane Discriminator için.
---

### Discriminator Kaybı

Discriminator iki şeyi doğru yapmalı:

1. Gerçek resimleri **gerçek** olarak tanımalı → etiket: `1`
2. Sahte resimleri **sahte** olarak tanımalı → etiket: `0`

```python
disc_loss = kayıp(gerçek=1,   discriminator(gerçek_resim))
          + kayıp(gerçek=0,   discriminator(sahte_resim))
```

### Generator Kaybı

Generator Discriminator'ı kandırmak istiyor.  
Yani Discriminator'ın ürettiği sahte resimlere "gerçek (1)" demesini istiyor:

```python
gen_loss = kayıp(gerçek=1,   discriminator(sahte_resim))
#                ↑
#          Generator sahtenin "gerçek" sayılmasını istiyor
```

### Label Smoothing: Eğitimi Dengeleyen Gizli Silah

Discriminator'a gerçek resimler için tam `1.0` değil, `0.9` veriyoruz.

Neden?  
Eğer `1.0` verirsek Discriminator çok keskin ve aşırı güvenli kararlar alır.  
Bu Generator'ı ezer — Generator öğrenemez.

```
Label Smoothing yok:  Gerçek = 1.0  → Discriminator aşırı güvenli → Generator ezilir
Label Smoothing var:  Gerçek = 0.9  → Discriminator biraz belirsiz → İkisi dengeli gelişir
```

Bu küçük değişiklik GAN kalitesini büyük oranda artırır.


In [ ]:
# Binary Crossentropy kayıp fonksiyonu
# from_logits=True: Discriminator'ın çıkışı sigmoid geçmeden geliyor (ham logit)
# Keras bunu içeride daha kararlı bir şekilde hesaplayacak
bce = tf.keras.losses.BinaryCrossentropy(from_logits=True)

GERCEK_ETIKET = 0.9   # Label smoothing: 1.0 yerine 0.9
SAHTE_ETIKET  = 0.0


def discriminator_loss(gercek_cikis, sahte_cikis):
    """Discriminator iki şeyi doğru yapmalı:
    1) Gerçekleri gerçek tanımalı
    2) Sahteleri sahte tanımalı
    """
    gercek_kayip = bce(tf.ones_like(gercek_cikis) * GERCEK_ETIKET, gercek_cikis)
    sahte_kayip  = bce(tf.zeros_like(sahte_cikis) + SAHTE_ETIKET,  sahte_cikis)
    return gercek_kayip + sahte_kayip


def generator_loss(sahte_cikis):
    """Generator Discriminator'ı kandırmak istiyor:
    Ürettiği sahte resimlere Discriminator 'gerçek (1)' desin.
    """
    return bce(tf.ones_like(sahte_cikis), sahte_cikis)


# Her model için ayrı optimizer — birbirinden bağımsız öğrenirler
gen_optimizer  = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
disc_optimizer = tf.keras.optimizers.Adam(learning_rate=2e-4, beta_1=0.5)
# beta_1=0.5: Normalde Adam 0.9 kullanır.
# GAN'larda 0.5 daha iyi çalışır — önceki gradyanları daha az hatırlar, daha çevik olur.

print("Kayıp fonksiyonları ve optimizer'lar hazır.")
print(f"Gerçek etiketi: {GERCEK_ETIKET} (Label Smoothing aktif)")
print(f"Sahte etiketi:  {SAHTE_ETIKET}")

---

## Adım 5: Eğitim Döngüsü

### Neden `model.fit()` Kullanamıyoruz?

FNN, CNN, RNN'de `model.fit()` yeterliydi çünkü tek bir model vardı.  
GAN'da iki model var ve birbirlerine bağlı özel bir eğitim mantığı gerekiyor.  
Bu yüzden eğitim döngüsünü **elle yazıyoruz**.

### Her Batch'te (Her 128 Resimde) Ne Olur?

```
① Generator 128 sahte resim üretir

② Discriminator değerlendirir:
   - 128 gerçek resme bakar → Bunlar gerçek (1) mi?
   - 128 sahte resme bakar → Bunlar sahte (0) mı?

③ Discriminator'ın kaybı hesaplanır ve ağırlıkları güncellenir
   (Generator'ın ağırlıkları DOKUNULMAZ)

④ Generator'ın kaybı hesaplanır ve ağırlıkları güncellenir
   (Discriminator'ın ağırlıkları DOKUNULMAZ)
```

### `GradientTape` Nedir?

TensorFlow'un "hangi ağırlık ne kadar değişmeli" sorusuna cevap veren araç.

```python
with tf.GradientTape() as tape:
    # Bu blok içindeki hesaplamalar izlenir
    kayip = model(giris)

gradyanlar = tape.gradient(kayip, model.trainable_variables)
# gradyanlar: her ağırlığın kaybı ne kadar artırdığını söyler

optimizer.apply_gradients(zip(gradyanlar, model.trainable_variables))
# Ağırlıkları güncelle: azaltmak için ters yönde adım at
```

### `@tf.function` Ne İşe Yarar?

Normalde Python kodu satır satır çalışır.  
`@tf.function` ile bu fonksiyon TensorFlow'un hızlı grafik sistemine derlenir.  
Sonuç: **3-5× daha hızlı eğitim!**


In [ ]:
# Sabit gürültü: eğitim boyunca aynı 20 vektörün gelişimini izleyeceğiz
# Bu sayede "epoch 1'de nasıl görünüyordu, epoch 50'de nasıl?" karşılaştırması yapabiliriz
SABIT_GURULTU = tf.random.normal([20, LATENT_DIM])


@tf.function   # Bu decorator fonksiyonu TF grafiğine derler → 3-5× hız
def egitim_adimi(gercek_resimler):
    """Tek bir batch için eğitim adımı."""

    # 128 tane rastgele gürültü vektörü
    gurultu = tf.random.normal([BATCH_SIZE, LATENT_DIM])

    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        # ① Generator sahte resimler üretir
        sahte_resimler = generator(gurultu, training=True)

        # ② Discriminator her ikisini değerlendirir
        gercek_cikis = discriminator(gercek_resimler, training=True)
        sahte_cikis  = discriminator(sahte_resimler,  training=True)

        # ③ Her iki kaybı hesapla
        g_kayip = generator_loss(sahte_cikis)
        d_kayip = discriminator_loss(gercek_cikis, sahte_cikis)

    # ④ Generator'ı güncelle (sadece Generator'ın ağırlıkları değişir)
    gen_gradyanlar = gen_tape.gradient(g_kayip, generator.trainable_variables)
    gen_optimizer.apply_gradients(zip(gen_gradyanlar, generator.trainable_variables))

    # ⑤ Discriminator'ı güncelle (sadece Discriminator'ın ağırlıkları değişir)
    disc_gradyanlar = disc_tape.gradient(d_kayip, discriminator.trainable_variables)
    disc_optimizer.apply_gradients(zip(disc_gradyanlar, discriminator.trainable_variables))

    return g_kayip, d_kayip


def gorsellestir(epoch, gen_kayip_gecmis, disc_kayip_gecmis):
    """Üretilen resimleri ve kayıp grafiğini göster."""
    tahminler = generator(SABIT_GURULTU, training=False).numpy()
    tahminler = np.clip((tahminler + 1) / 2, 0, 1)   # [-1,+1] → [0,1]

    fig = plt.figure(figsize=(22, 6))

    # 20 üretilen resim — 2 satır × 10 sütun
    for i in range(20):
        ax = fig.add_subplot(2, 13, i + 1 if i < 10 else i + 4)
        ax.imshow(tahminler[i])
        ax.axis('off')

    # Kayıp grafiği
    ax_k = fig.add_subplot(1, 4, 4)
    ax_k.plot(gen_kayip_gecmis,  color='#2196F3', linewidth=2, label='Generator Kaybı')
    ax_k.plot(disc_kayip_gecmis, color='#FF5722', linewidth=2, linestyle='--', label='Discriminator Kaybı')
    ax_k.axhline(0.693, color='gray', linestyle=':', alpha=0.7, label='İdeal denge (~0.69)')
    ax_k.set_title('Kayıp Geçmişi')
    ax_k.set_xlabel('Epoch')
    ax_k.legend(fontsize=8)
    ax_k.grid(True, alpha=0.3)

    fig.suptitle(f'Epoch {epoch} — GAN\'ın Ürettiği Renkli Giysiler',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


print("Eğitim fonksiyonları hazır!")

---

## Adım 6: Eğitimi Başlat!

### Kayıp Grafiğini Nasıl Yorumlarsın?

```
İdeal senaryo:
Generator Kaybı    ≈ 0.69  (log 2 ≈ 0.693)
Discriminator Kaybı ≈ 0.69

İkisi de 0.69 civarında geziniyorsa → İki model dengeli → Eğitim sağlıklı
```

**Kötü senaryolar:**

- Discriminator Kaybı → 0: Discriminator çok güçlendi, Generator öğrenemiyor
- Generator Kaybı → çok yüksek: Generator Discriminator'ı hiç kandıramıyor
- İkisi de sallanıyorsa: Normal! GAN eğitimi doğası gereği kararsızdır.

### Ne Zaman Ne Göreceksin?

```
Epoch  1  → Renkli gürültü (anlamsız lekeler)
Epoch  5  → Renkler ayrışmaya başlar
Epoch 10  → Giysi şekilleri belirir
Epoch 20  → Silüetler daha net
Epoch 50  → Tanınabilir renkli giysiler
Epoch 100 → En iyi sonuç
```


## Canlı 3D görselleştirme — GeneratorGAN'in kendi eğitim döngüsü var (`model.fit()` çağrılmıyor), bu yüzden`nn3d.Monitor` callback'i burada işe yaramaz. Onun yerine döngünün içinden`gorunum.update(...)` ile her epoch bir kare gönderiyoruz.Generator'da izlemeye değer olan: `Dense(25088)` → `Reshape(7,7,512)` →`Conv2DTranspose` zinciri. Gürültü vektörünün nasıl adım adım genişleyipresme dönüştüğünü katman kartlarındaki tensör şekillerinden takip edebilirsin(`7x7x512` → `14x14x256` → `28x28x128` → `28x28x3`).

In [ ]:
EPOCHS  = 100
GOSTER  = {1, 5, 10, 20, 30, 50, 75, 100}   # Bu epoch'larda görsel göster

gen_kayip_gecmis  = []
disc_kayip_gecmis = []

print(f"Eğitim başlıyor: {EPOCHS} epoch, batch başına {BATCH_SIZE} resim")
print(f"Her epoch: {len(dataset)} batch × {BATCH_SIZE} resim = {len(dataset)*BATCH_SIZE} resim")
print("=" * 65)

baslangic = time.time()

for epoch in range(1, EPOCHS + 1):
    epoch_gen, epoch_disc = [], []

    for batch in dataset:
        g_k, d_k = egitim_adimi(batch)
        epoch_gen.append(float(g_k))
        epoch_disc.append(float(d_k))

    gen_kayip_gecmis.append(np.mean(epoch_gen))
    disc_kayip_gecmis.append(np.mean(epoch_disc))

    gecen_dk = (time.time() - baslangic) / 60
    print(
        f"Epoch {epoch:3d}/{EPOCHS} | "
        f"Gen Kaybı: {gen_kayip_gecmis[-1]:.4f} | "
        f"Disc Kaybı: {disc_kayip_gecmis[-1]:.4f} | "
        f"Geçen: {gecen_dk:.1f} dk"
    )

    # nn3d: her epoch sonunda Generator'un o anki durumunu yolla.
    gorunum.update(
        SABIT_GURULTU[:1].numpy(),
        epoch=epoch,
        metrics={'gen_kayip': gen_kayip_gecmis[-1],
                 'disc_kayip': disc_kayip_gecmis[-1]},
    )

    if epoch in GOSTER:
        gorsellestir(epoch, gen_kayip_gecmis, disc_kayip_gecmis)

print(f"\nEğitim tamamlandı! Toplam süre: {(time.time()-baslangic)/60:.1f} dakika")

---

## Adım 7: Sonuçları İncele


In [ ]:
# 80 yeni, hiç var olmamış giysi tasarımı üret
yeni_gurultu = tf.random.normal([80, LATENT_DIM])
uretilen     = generator(yeni_gurultu, training=False).numpy()
uretilen     = np.clip((uretilen + 1) / 2, 0, 1)

fig, axes = plt.subplots(8, 10, figsize=(20, 16))
for i, ax in enumerate(axes.flat):
    ax.imshow(uretilen[i])
    ax.axis('off')

plt.suptitle(f'GAN Tarafından Üretilen 80 Yeni Renkli Giysi Tasarımı ({EPOCHS} Epoch)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Gerçek vs GAN karşılaştırması — yan yana
n = 10
idx       = np.random.choice(len(X_color), n, replace=False)
gercekler = np.clip((X_color[idx] + 1) / 2, 0, 1)
sahteler  = np.clip(
    (generator(tf.random.normal([n, LATENT_DIM]), training=False).numpy() + 1) / 2,
    0, 1
)

fig, axes = plt.subplots(2, n, figsize=(22, 5))
for i in range(n):
    axes[0, i].imshow(gercekler[i])
    axes[0, i].axis('off')
    axes[1, i].imshow(sahteler[i])
    axes[1, i].axis('off')

axes[0, 0].set_ylabel('GERÇEK', fontsize=12, fontweight='bold', color='green', labelpad=12)
axes[1, 0].set_ylabel('GAN\nÜRETTİ', fontsize=12, fontweight='bold', color='crimson', labelpad=12)

plt.suptitle('Gerçek Renkli Giysiler vs GAN\'ın Ürettiği Giysiler',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---

## Adım 8: Latent Space — Gizli Uzay Keşfi

### Latent Space Nedir?

Generator 256 boyutlu bir uzayda çalışıyor. Bu uzayın her noktası bir giysi tasarımına karşılık geliyor.

### İnterpolasyon — İki Tasarım Arasında Geçiş

İki farklı vektör seç: A noktası ve B noktası.  
İkisi arasında küçük küçük adımlarla geç.

```
A (Kırmızı t-shirt) ──────────────────── B (Mavi pantolon)
    ↓                                        ↓
α=0.0  α=0.1  α=0.2  ...  α=0.8  α=0.9  α=1.0
```

Eğer GAN gerçekten anlamlı bir uzay öğrendiyse geçiş **pürüzsüz ve mantıklı** olur.  
Birdenbire bambaşka bir şeye zıplamazlar — bu GAN'ın başarısının kanıtıdır.


In [ ]:
n_adim = 14   # A'dan B'ye kaç adımda gidecek

z_a = tf.random.normal([1, LATENT_DIM])   # Başlangıç tasarımı
z_b = tf.random.normal([1, LATENT_DIM])   # Bitiş tasarımı

# İki vektör arasında lineer interpolasyon
# alpha=0 → tamamen A, alpha=1 → tamamen B
alphalar   = np.linspace(0, 1, n_adim)
ara_noktalar = tf.concat([(1 - a) * z_a + a * z_b for a in alphalar], axis=0)

ara_resimler = generator(ara_noktalar, training=False).numpy()
ara_resimler = np.clip((ara_resimler + 1) / 2, 0, 1)

fig, axes = plt.subplots(1, n_adim, figsize=(24, 3))
for i, ax in enumerate(axes):
    ax.imshow(ara_resimler[i])
    ax.set_title(f'α={alphalar[i]:.2f}', fontsize=7)
    ax.axis('off')

plt.suptitle(
    'Latent Space İnterpolasyonu\n'
    'Sol (α=0): Tasarım A  →  Sağ (α=1): Tasarım B  →  Arası: Pürüzsüz Geçiş',
    fontsize=11, fontweight='bold'
)
plt.tight_layout()
plt.show()

print("Geçiş pürüzsüzse → GAN anlamlı bir uzay öğrenmiş demektir!")
print("Geçiş bozuk/dağınıksa → Daha fazla epoch gerekiyor.")

In [ ]:
# Modeli kaydet — bir daha eğitmek zorunda kalmayasın
generator.save('gan_renkli_generator.keras')
print("Generator kaydedildi → gan_renkli_generator.keras")
print()
print("Daha sonra tekrar kullanmak için:")
print("  import tensorflow as tf, numpy as np, matplotlib.pyplot as plt")
print("  gen = tf.keras.models.load_model('gan_renkli_generator.keras')")
print("  gurultu = tf.random.normal([16, 256])")
print("  resimler = gen(gurultu, training=False).numpy()")
print("  plt.imshow(np.clip((resimler[0] + 1) / 2, 0, 1))")

---

## Özet: Ne Öğrendik?

### GAN'ın Temel Mantığı

```
Generator (Sahtekâr)  ←→  Discriminator (Dedektif)
     ↓                           ↓
Daha iyi sahte üret       Daha iyi yakala
     ↓                           ↓
           Sonunda...
           Generator o kadar iyi olur ki
           Discriminator ayırt edemez
           → Gerçekçi giysiler üretilir!
```

### Bu Notebook'ta Kullandığımız Teknikler

| Teknik                 | Ne İşe Yarar?                                                  |
| ---------------------- | -------------------------------------------------------------- |
| **Conv2DTranspose**    | Küçük özellik haritasını büyük resme dönüştürür (Generator'da) |
| **LeakyReLU**          | Negatif değerlere küçük eğim verir → nöronlar ölmez            |
| **BatchNormalization** | Değerleri normalize eder → eğitim kararlı kalır                |
| **Dropout**            | Discriminator'ı yavaşlatır → iki model dengeli gelişir         |
| **Label Smoothing**    | Discriminator aşırı güvenmez → Generator daha iyi öğrenir      |
| **GradientTape**       | İki modelin gradyanlarını ayrı ayrı hesaplar                   |
| **Latent Space**       | 256 boyutlu uzayda her nokta bir tasarıma karşılık gelir       |
| **İnterpolasyon**      | İki tasarım arasında pürüzsüz geçiş → GAN kalitesinin kanıtı   |

### GAN'ın Güçlü ve Zayıf Yönleri

| Güçlü Yönler                | Zayıf Yönler                                |
| --------------------------- | ------------------------------------------- |
| Yoktan resim üretir         | Eğitmesi zor ve dengesiz                    |
| Sınırsız çeşitlilik         | Mode collapse riski (hep aynı resmi üretme) |
| Latent space keşfedilebilir | Sonuç kalitesi garanti değil                |

### Sonraki Adımlar

| Model                | Ne Yapar?                                                        |
| -------------------- | ---------------------------------------------------------------- |
| **Conditional GAN**  | "Sadece kırmızı t-shirt üret" — kategoriye göre kontrollü üretim |
| **StyleGAN**         | Gerçekçi insan yüzü üretimi (NVIDIA'nın ünlü modeli)             |
| **Pix2Pix**          | Elle çizimden gerçekçi fotoğraf üretimi                          |
| **CycleGAN**         | At fotoğrafını zebra fotoğrafına çevir — ikili dönüşüm           |
| **Stable Diffusion** | Metin → resim ("kırmızı spor ayakkabı, beyaz zemin")             |
